### Power Flow Results Viewer: Geospatial Edition

In [7]:
def add_calculated_results(net):
    """
    Add calculated power flow result columns to res_bus and res_line dataframes.

    Multiconductor uses MultiIndex (element_idx, phase) where phases are:
    1=A, 2=B, 3=C, 4=N (neutral)

    This function pivots the per-phase data into separate columns matching
    the full output specification.
    """
    import numpy as np
    import pandas as pd

    phase_map = {1: 'A', 2: 'B', 3: 'C', 4: 'N'}

    # ---------------------------------------------------------------------------
    # Helper: get vn_kv for a bus index from net.bus  (needed for current calc)
    # ---------------------------------------------------------------------------
    def _vn_kv_for_bus(bus_idx):
        """Return nominal voltage in kV for a given bus index."""
        try:
            bus_df = net.bus if isinstance(net.bus, pd.DataFrame) else None
            if bus_df is not None and 'vn_kv' in bus_df.columns:
                if isinstance(bus_df.index, pd.MultiIndex):
                    # first level is the bus id
                    vals = bus_df.loc[bus_idx, 'vn_kv']
                    return float(vals.iloc[0]) if hasattr(vals, 'iloc') else float(vals)
                elif bus_idx in bus_df.index:
                    return float(bus_df.at[bus_idx, 'vn_kv'])
        except Exception:
            pass
        return np.nan

    # ===== Process res_bus =====
    if hasattr(net, 'res_bus') and net.res_bus is not None and len(net.res_bus) > 0:
        res_bus = net.res_bus.copy()

        if isinstance(res_bus.index, pd.MultiIndex):
            bus_indices = res_bus.index.get_level_values(0).unique()

            result_rows = []
            for bus_idx in bus_indices:
                row = {'bus_idx': bus_idx}

                try:
                    bus_data = res_bus.loc[bus_idx]
                except KeyError:
                    continue

                vn_kv = _vn_kv_for_bus(bus_idx)
                vm_values = {}
                va_values = {}

                for phase_num, phase_letter in phase_map.items():
                    try:
                        if phase_num not in bus_data.index:
                            continue
                        phase_data = bus_data.loc[phase_num]

                        # --- Voltage magnitude & angle ---
                        vm = phase_data.get('vm_pu', np.nan)
                        va = phase_data.get('va_degree', np.nan)

                        if pd.notna(vm):
                            row[f'SVVOLTAGE_V_{phase_letter}_PU'] = vm
                            vm_values[phase_letter] = vm
                        if pd.notna(va):
                            row[f'SVVOLTAGE_{phase_letter}_ANGLE'] = va
                            va_values[phase_letter] = va

                        # --- Total power at bus per phase ---
                        p_val = phase_data.get('p_mw', np.nan)
                        q_val = phase_data.get('q_mvar', np.nan)

                        if pd.notna(p_val):
                            row[f'SVPOWERFLOW_P_{phase_letter}_MW'] = p_val
                        if pd.notna(q_val):
                            row[f'SVPOWERFLOW_Q_{phase_letter}_MVAR'] = q_val

                        # --- Load / Gen split (negative P/Q = load) ---
                        if pd.notna(p_val):
                            if p_val < 0:
                                row[f'SVPOWERFLOW_P_LOAD_{phase_letter}_MW'] = abs(p_val)
                                row[f'SVPOWERFLOW_P_GEN_{phase_letter}_MW'] = 0.0
                            else:
                                row[f'SVPOWERFLOW_P_LOAD_{phase_letter}_MW'] = 0.0
                                row[f'SVPOWERFLOW_P_GEN_{phase_letter}_MW'] = p_val

                        if pd.notna(q_val):
                            if q_val < 0:
                                row[f'SVPOWERFLOW_Q_LOAD_{phase_letter}_MVAR'] = abs(q_val)
                                row[f'SVPOWERFLOW_Q_GEN_{phase_letter}_MVAR'] = 0.0
                            else:
                                row[f'SVPOWERFLOW_Q_LOAD_{phase_letter}_MVAR'] = 0.0
                                row[f'SVPOWERFLOW_Q_GEN_{phase_letter}_MVAR'] = q_val

                        # --- Bus current magnitude & angle ---
                        # I = S* / V  →  |I| = |S| / |V|,  angle(I) = -atan2(Q,P) - va_rad
                        if pd.notna(p_val) and pd.notna(q_val) and pd.notna(vm) and pd.notna(vn_kv) and vm * vn_kv > 0:
                            s_mag = np.sqrt(p_val ** 2 + q_val ** 2)  # MVA
                            i_ka = s_mag / (vm * vn_kv)               # kA
                            row[f'SVCURRENT_CURRENT_{phase_letter}_KA'] = i_ka

                            if pd.notna(va):
                                va_rad = np.deg2rad(va)
                                s_angle = np.arctan2(q_val, p_val)    # angle of S
                                i_angle = np.rad2deg(-s_angle - va_rad)
                                row[f'SVCURRENT_{phase_letter}_ANGLE'] = i_angle

                    except (KeyError, TypeError):
                        pass

                # --- Aggregate voltage stats (mean of A, B, C) ---
                abc_vm = [vm_values.get(p) for p in ['A', 'B', 'C'] if vm_values.get(p) is not None]
                abc_va = [va_values.get(p) for p in ['A', 'B', 'C'] if va_values.get(p) is not None]
                if abc_vm:
                    row['SVVOLTAGE_VM_PU'] = np.mean(abc_vm)
                if abc_va:
                    row['SVVOLTAGE_VM_DEG'] = np.mean(abc_va)

                # --- Voltage unbalance (IEEE definition) ---
                if len(abc_vm) >= 2:
                    vm_avg = np.mean(abc_vm)
                    if vm_avg > 0:
                        vm_max_dev = max(abs(v - vm_avg) for v in abc_vm)
                        row['UNBALANCEVOLTAGEPERCENTAGE'] = (vm_max_dev / vm_avg) * 100
                    else:
                        row['UNBALANCEVOLTAGEPERCENTAGE'] = 0.0

                result_rows.append(row)

            # Broadcast calculated columns back to MultiIndex res_bus
            if result_rows:
                calc_df = pd.DataFrame(result_rows).set_index('bus_idx')
                for col in calc_df.columns:
                    net.res_bus[col] = net.res_bus.index.get_level_values(0).map(
                        calc_df[col].to_dict()
                    ).values

    # ===== Process res_line =====
    if hasattr(net, 'res_line') and net.res_line is not None and len(net.res_line) > 0:
        res_line = net.res_line.copy()

        if isinstance(res_line.index, pd.MultiIndex):
            line_indices = res_line.index.get_level_values(0).unique()

            # Look up max_i_ka per line from net.line or line_std_types for loading %
            def _max_i_ka_for_line(line_idx):
                try:
                    line_df = net.line if isinstance(net.line, pd.DataFrame) else None
                    if line_df is not None and 'max_i_ka' in line_df.columns and line_idx in line_df.index:
                        return float(line_df.at[line_idx, 'max_i_ka'])
                except Exception:
                    pass
                return np.nan

            result_rows = []
            for line_idx in line_indices:
                row = {'line_idx': line_idx}

                try:
                    line_data = res_line.loc[line_idx]
                except KeyError:
                    continue

                max_i = _max_i_ka_for_line(line_idx)

                for phase_num, phase_letter in phase_map.items():
                    try:
                        if phase_num not in line_data.index:
                            continue
                        phase_data = line_data.loc[phase_num]

                        # --- Power flows ---
                        if 'p_from_mw' in phase_data.index:
                            row[f'SVPOWERFLOW_P_FROMBUS_{phase_letter}_MW'] = phase_data['p_from_mw']
                        if 'q_from_mvar' in phase_data.index:
                            row[f'SVPOWERFLOW_Q_FROMBUS_{phase_letter}_MVAR'] = phase_data['q_from_mvar']
                        if 'p_to_mw' in phase_data.index:
                            row[f'SVPOWERFLOW_P_TOBUS_{phase_letter}_MW'] = phase_data['p_to_mw']
                        if 'q_to_mvar' in phase_data.index:
                            row[f'SVPOWERFLOW_Q_TOBUS_{phase_letter}_MVAR'] = phase_data['q_to_mvar']
                        if 'pl_mw' in phase_data.index:
                            row[f'SVPOWERFLOW_P_LOSS_{phase_letter}_MW'] = phase_data['pl_mw']
                        if 'ql_mvar' in phase_data.index:
                            row[f'SVPOWERFLOW_Q_LOSS_{phase_letter}_MVAR'] = phase_data['ql_mvar']

                        # --- Current magnitudes ---
                        if 'i_from_ka' in phase_data.index:
                            i_from = phase_data['i_from_ka']
                            row[f'SVCURRENT_CURRENT_FROMBUS_{phase_letter}_KA'] = i_from
                        if 'i_to_ka' in phase_data.index:
                            i_to = phase_data['i_to_ka']
                            row[f'SVCURRENT_CURRENT_TOBUS_{phase_letter}_KA'] = i_to

                        # --- Loading percent per phase ---
                        if 'loading_percent' in phase_data.index:
                            row[f'SVPOWERFLOW_PERCENT_LOAD_{phase_letter}'] = phase_data['loading_percent']
                        elif pd.notna(max_i) and max_i > 0:
                            # Derive from from-side current
                            i_val = row.get(f'SVCURRENT_CURRENT_FROMBUS_{phase_letter}_KA', np.nan)
                            if pd.notna(i_val):
                                row[f'SVPOWERFLOW_PERCENT_LOAD_{phase_letter}'] = (i_val / max_i) * 100

                    except (KeyError, TypeError):
                        pass

                result_rows.append(row)

            if result_rows:
                calc_df = pd.DataFrame(result_rows).set_index('line_idx')
                for col in calc_df.columns:
                    net.res_line[col] = net.res_line.index.get_level_values(0).map(
                        calc_df[col].to_dict()
                    ).values

    return net

In [1]:
#%pip install keplergl
from keplergl import KeplerGl
import multiconductor as mc
import pandas as pd
import warnings
warnings.filterwarnings(action="ignore", category=RuntimeWarning)
warnings.filterwarnings(action="ignore", category=UserWarning)
from multiconductor.pycci.cci_powerflow import run_pf
from multiconductor.pycci.cci_powerflow import run_pf
from multiconductor.load_allocation.load_allocation import (
    build_measurement_graph,
    connected_capacity_allocation,
    get_rated_S,
 )
import pickle
import numpy as np
import plotly.express as px

CKT_114_16955 = pickle.load(open('networks/new/mc/CKT_114_16955.pkl', 'rb'))
run_pf(CKT_114_16955, run_control=False)
CKT_4799_01085 = pickle.load(open('networks/new/mc/CKT_4799_01085.pkl', 'rb'))
run_pf(CKT_4799_01085, run_control=False)

CKT_114_16955_geography = pd.read_csv('CKT_114_16955_geography.csv')
CKT_4799_01085_geography = pd.read_csv('CKT_4799_01085_geography.csv')

import ast

def import_geocoords(net, geography_df):
    """Import geographic coordinates from geography DataFrame into pandapower net bus_geodata and line_geodata."""
    geo = geography_df[['CONDUCTING_EQUIPMENTID', 'CONNECTIVITY_NODEID', 'GEOMETRY_COORDINATES']].dropna(subset=['GEOMETRY_COORDINATES']).copy()
    geo['parsed'] = geo['GEOMETRY_COORDINATES'].apply(ast.literal_eval)
    geo['geom_type'] = geo['parsed'].apply(lambda c: c.get('type'))
    
    # === Bus geodata via CONNECTIVITY_NODEID on Point geometries ===
    points = geo[geo['geom_type'] == 'Point'].copy()
    points['lon'] = points['parsed'].apply(lambda c: c['coordinates'][0])
    points['lat'] = points['parsed'].apply(lambda c: c['coordinates'][1])
    points['node_id'] = points['CONNECTIVITY_NODEID'].astype(str)
    # Filter valid nodes and deduplicate
    valid_points = points[points['node_id'] != '-999999'].drop_duplicates(subset='node_id')
    node_lookup = valid_points.set_index('node_id')[['lat', 'lon']]
    
    bus_geo = net.bus[['name']].copy()
    bus_geo = bus_geo.join(node_lookup, on='name')
    matched_buses = bus_geo.dropna(subset=['lat', 'lon'])
    net.bus_geodata = pd.DataFrame(index=matched_buses.index)
    net.bus_geodata['Longitude'] = matched_buses['lon'].values
    net.bus_geodata['Latitude'] = matched_buses['lat'].values

    # === Line geodata via CONDUCTING_EQUIPMENTID ===
    linestrings = geo[geo['geom_type'] == 'LineString'].copy()
    line_coord_lookup = {}
    for _, row in linestrings.iterrows():
        coords_list = [(c[0], c[1]) for c in row['parsed']['coordinates']]
        line_coord_lookup[row['CONDUCTING_EQUIPMENTID']] = coords_list

    line_geodata_records = {}
    for idx, row in net.line.iterrows():
        if row['name'] in line_coord_lookup:
            line_geodata_records[idx] = line_coord_lookup[row['name']]
        else:
            from_bus, to_bus = row['from_bus'], row['to_bus']
            coords = []
            if from_bus in net.bus_geodata.index:
                coords.append((net.bus_geodata.at[from_bus, 'Longitude'], net.bus_geodata.at[from_bus, 'Latitude']))
            if to_bus in net.bus_geodata.index:
                coords.append((net.bus_geodata.at[to_bus, 'Longitude'], net.bus_geodata.at[to_bus, 'Latitude']))
            if len(coords) == 2:
                line_geodata_records[idx] = coords

    net.line_geodata = pd.DataFrame({'Geometry': pd.Series(line_geodata_records)})

    print(f"Bus geodata:  {len(net.bus_geodata)}/{len(net.bus)} matched")
    print(f"Line geodata: {len(net.line_geodata)}/{len(net.line)} matched")

print("=== CKT_114_16955 ===")
import_geocoords(CKT_114_16955, CKT_114_16955_geography)
print("\n=== CKT_4799_01085 ===")
import_geocoords(CKT_4799_01085, CKT_4799_01085_geography)

def get_bus_element_multimap(net):
    """Build bus_index -> set of element_types (a bus can belong to multiple element types)."""
    from collections import defaultdict
    bus_elems = defaultdict(set)
    element_types = ['ext_grid_sequence', 'trafo', 'trafo3w', 'trafo1ph', 'asymmetric_shunt', 'asymmetric_load', 'asymmetric_sgen', 'switch']
    for elem in element_types:
        if elem in net and hasattr(net[elem], 'columns') and len(net[elem]) > 0:
            df = net[elem]
            # trafo1ph stores hv_bus in index level 'bus'
            if elem == 'trafo1ph' and 'bus' in df.index.names:
                for bus_idx in set(df.index.get_level_values('bus')):
                    bus_elems[bus_idx].add(elem)
            else:
                bus_cols = [c for c in df.columns if c == 'bus' or c in ('hv_bus', 'lv_bus', 'mv_bus')]
                for col in bus_cols:
                    for bus_idx in df[col]:
                        bus_elems[bus_idx].add(elem)
    return bus_elems

# Color palette: RGB per element type
ELEMENT_RGB = {
    'bus':               [255, 0, 0],      # red
    'ext_grid_sequence': [255, 255, 0],    # yellow
    'switch':            [0, 150, 255],    # blue
    'asymmetric_shunt':  [255, 0, 255],    # magenta
    'asymmetric_load':   [0, 255, 150],    # green-cyan
    'asymmetric_sgen':   [255, 128, 200],  # pink
    'trafo':             [255, 165, 0],    # orange
    'trafo3w':           [255, 165, 0],    # orange
    'trafo1ph':          [255, 165, 0],    # orange
}

RES_BUS_COLS = ['vm_pu', 'va_degree', 'p_mw', 'q_mvar', 'imbalance_percent']
RES_LINE_COLS = ['i_from_ka', 'i_to_ka', 'i_ka', 'p_from_mw', 'p_to_mw',
                 'q_from_mvar', 'q_to_mvar', 'pl_mw', 'ql_mvar', 'loading_percent']

def build_kepler_map(net, title="Network"):
    bus_elems = get_bus_element_multimap(net)

    # Bus geodata has MultiIndex (bus_number, phase)
    bus_gdf = net.bus_geodata.copy()

    # Join res_bus fields (same MultiIndex)
    res_bus_cols = [c for c in RES_BUS_COLS if c in net.res_bus.columns]
    if len(res_bus_cols) > 0 and len(net.res_bus) > 0:
        bus_gdf = bus_gdf.join(net.res_bus[res_bus_cols], how='left')

    # Also add bus name for tooltip
    if 'name' in net.bus.columns:
        bus_gdf = bus_gdf.join(net.bus[['name']].rename(columns={'name': 'bus_name'}), how='left')

    # Columns to keep for each bus subset
    keep_cols = ['Longitude', 'Latitude']
    if 'bus_name' in bus_gdf.columns:
        keep_cols.append('bus_name')
    keep_cols.extend([c for c in res_bus_cols if c in bus_gdf.columns])

    # Build per-element-type bus sets (a bus can appear in multiple layers)
    from collections import defaultdict
    etype_bus_indices = defaultdict(set)
    for idx in bus_gdf.index:
        bus_num = idx[0]
        etypes = bus_elems.get(bus_num, set())
        if not etypes:
            etype_bus_indices['bus'].add(idx)
        else:
            for et in etypes:
                etype_bus_indices[et].add(idx)

    data_dict = {}
    layers = []
    bus_tooltip_fields = [{'name': c, 'format': None} for c in keep_cols if c not in ('Longitude', 'Latitude')]

    for etype in sorted(etype_bus_indices.keys()):
        indices = sorted(etype_bus_indices[etype])
        subset = bus_gdf.loc[bus_gdf.index.isin(indices), keep_cols].copy()
        if len(subset) == 0:
            continue
        data_name = etype
        data_dict[data_name] = subset
        color = ELEMENT_RGB.get(etype, [255, 0, 0])
        layers.append({
            'id': f'{etype}_layer',
            'type': 'point',
            'config': {
                'dataId': data_name,
                'label': etype,
                'color': color,
                'columns': {'lat': 'Latitude', 'lng': 'Longitude'},
                'isVisible': True,
                'visConfig': {
                    'radius': 5,
                    'opacity': 0.8,
                }
            }
        })
        print(f"  {etype}: {len(subset)} points, color={color}")

    # Build line data with res_line fields
    line_gdf = net.line_geodata.copy()
    # Align index names with res_line for join compatibility
    line_gdf.index.names = net.res_line.index.names
    line_gdf['Geometry'] = line_gdf['Geometry'].apply(
        lambda coords: 'LINESTRING (' + ', '.join(f'{c[0]} {c[1]}' for c in coords) + ')' if len(coords) >= 2 else None
    )
    line_gdf = line_gdf.dropna(subset=['Geometry'])

    # Join res_line fields
    res_line_cols = [c for c in RES_LINE_COLS if c in net.res_line.columns]
    if len(res_line_cols) > 0 and len(net.res_line) > 0:
        line_gdf = line_gdf.join(net.res_line[res_line_cols], how='left')

    # Add line name for tooltip
    if 'name' in net.line.columns:
        line_gdf = line_gdf.join(net.line[['name']].rename(columns={'name': 'line_name'}), how='left')

    data_dict['lines'] = line_gdf

    line_tooltip_fields = []
    if 'line_name' in line_gdf.columns:
        line_tooltip_fields.append({'name': 'line_name', 'format': None})
    line_tooltip_fields.extend([{'name': c, 'format': None} for c in res_line_cols if c in line_gdf.columns])

    layers.append({
        'id': 'lines_layer',
        'type': 'geojson',
        'config': {
            'dataId': 'lines',
            'label': 'lines',
            'color': [0, 255, 0],
            'columns': {'geojson': 'Geometry'},
            'isVisible': True,
            'visConfig': {
                'strokeColor': [0, 255, 0],
                'thickness': 2,
                'opacity': 0.8
            }
        }
    })

    # Build tooltip config: fieldsToShow per dataId
    fields_to_show = {}
    for etype in etype_bus_indices:
        fields_to_show[etype] = bus_tooltip_fields
    fields_to_show['lines'] = line_tooltip_fields

    cfg = {
        'version': 'v1',
        'config': {
            'visState': {'layers': layers},
            'mapState': {
                'latitude': float(net.bus_geodata['Latitude'].mean()),
                'longitude': float(net.bus_geodata['Longitude'].mean()),
                'zoom': 13
            },
            'interactionConfig': {
                'tooltip': {
                    'enabled': True,
                    'fieldsToShow': fields_to_show
                }
            }
        }
    }

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kmap = KeplerGl(height=800, data=data_dict, config=cfg)
    return kmap



c:\Users\fgonzales\AppData\Local\anaconda3\envs\cympy\Lib\site-packages\keplergl\keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


=== CKT_114_16955 ===
Bus geodata:  1416/1816 matched
Line geodata: 360/360 matched

=== CKT_4799_01085 ===
Bus geodata:  4260/5240 matched
Line geodata: 750/750 matched


In [2]:
run_pf(CKT_114_16955, run_control=False)
st_charles_map = build_kepler_map(CKT_114_16955, "CKT_114_16955")
st_charles_map


  asymmetric_shunt: 20 points, color=[255, 0, 255]
  bus: 248 points, color=[255, 0, 0]
  switch: 872 points, color=[0, 150, 255]
  trafo1ph: 292 points, color=[255, 165, 0]
User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'visState': {'layers': [{'id': 'asymmetric_shunt_layer', 'type': …

In [3]:
run_pf(CKT_4799_01085, run_control=False)
barcelona_map = build_kepler_map(CKT_4799_01085, "CKT_4799_01085")
barcelona_map

  asymmetric_shunt: 12 points, color=[255, 0, 255]
  bus: 1664 points, color=[255, 0, 0]
  switch: 2472 points, color=[0, 150, 255]
  trafo1ph: 664 points, color=[255, 165, 0]
User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'visState': {'layers': [{'id': 'asymmetric_shunt_layer', 'type': …

### Voltage (p.u.) vs Distance from Source

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Build a map: bus_id -> set of element letter codes
bus_labels = {}
THRESHOLD = 0.1  # annotate buses with vm_pu < this

# Transformers (bus is MultiIndex level)
tp = net_test.trafo1ph
blv = tp.index.names.index("bus")
for full_idx in tp.index:
    b = full_idx[blv]
    bus_labels.setdefault(b, set()).add("T")

# Lines (from_bus, to_bus columns)
if hasattr(net_test, 'line') and not net_test.line.empty:
    for col in ('from_bus', 'to_bus'):
        if col in net_test.line.columns:
            for b in net_test.line[col].unique():
                bus_labels.setdefault(b, set()).add("Li")

# Shunt elements
for tbl_name, letter in [('asymmetric_load', 'L'), ('asymmetric_sgen', 'G'),
                          ('ext_grid', 'E'), ('asymmetric_shunt', 'Sh'),
                          ('switch', 'S')]:
    tbl = getattr(net_test, tbl_name, None)
    if tbl is not None and not tbl.empty and 'bus' in tbl.columns:
        for b in tbl['bus'].unique():
            bus_labels.setdefault(b, set()).add(letter)

# Build annotation data for near-zero buses
res = net_test.res_bus['vm_pu'].dropna()
if isinstance(res.index, pd.MultiIndex):
    bus_ids = res.index.get_level_values("index")
else:
    bus_ids = res.index

ann_x, ann_y, ann_text = [], [], []
for i, (bus_id, v) in enumerate(zip(bus_ids, res.values)):
    if v < THRESHOLD:
        label = ",".join(sorted(bus_labels.get(bus_id, {"?"})))
        ann_x.append(i)
        ann_y.append(v)
        ann_text.append(label)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(vm_pu_new))), y=vm_pu_new.values,
                         mode='markers', name='vm_pu', marker=dict(size=3, opacity=0.5)))
# Annotated near-zero points
fig.add_trace(go.Scatter(x=ann_x, y=ann_y, mode='markers+text', name='near-zero elements',
                         text=ann_text, textposition='top center',
                         textfont=dict(size=9, color='yellow'),
                         marker=dict(size=6, color='red', symbol='x')))
fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.add_hline(y=0.95, line_dash="dot", line_color="red")
fig.add_hline(y=1.05, line_dash="dot", line_color="red")
fig.update_layout(template="plotly_dark", height=600,
                  title="vm_pu with element annotations (E=ext_grid, G=sgen, L=load, Li=line, S=switch, Sh=shunt, T=trafo)",
                  xaxis_title="Bus Index", yaxis_title="vm_pu")
fig.show()

In [18]:
from collections import defaultdict, deque
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    """Distance in km between two (lat, lon) points."""
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

def build_bus_coords(net):
    """Build dict bus_num -> (lat, lon) from bus_geodata, handling MultiIndex."""
    coords = {}
    for idx in net.bus_geodata.index:
        bus_num = idx[0] if isinstance(idx, tuple) else idx
        if bus_num not in coords:
            coords[bus_num] = (
                net.bus_geodata.at[idx, 'Latitude'],
                net.bus_geodata.at[idx, 'Longitude'],
            )
    return coords

def line_geodesic_length(net, line_idx, bus_coords):
    """Compute geodesic length of a line from its geodata coordinates, fallback to bus endpoints."""
    if line_idx in net.line_geodata.index:
        coords = net.line_geodata.at[line_idx, 'Geometry']
        if isinstance(coords, list) and len(coords) >= 2:
            dist = 0.0
            for i in range(len(coords) - 1):
                dist += haversine(coords[i][1], coords[i][0], coords[i+1][1], coords[i+1][0])
            return dist
    # Fallback: straight-line from bus to bus
    row = net.line.loc[line_idx]
    fb, tb = row['from_bus'], row['to_bus']
    if fb in bus_coords and tb in bus_coords:
        la, lo = bus_coords[fb]
        lb, lb_lon = bus_coords[tb]
        return haversine(la, lo, lb, lb_lon)
    return 0.0

def compute_distance_from_source(net):
    """BFS from source bus through lines, trafos, and switches to compute cumulative distance (km) per bus."""
    bus_coords = build_bus_coords(net)
    adj = defaultdict(list)

    # Add line edges
    for idx, row in net.line.iterrows():
        fb, tb = row['from_bus'], row['to_bus']
        adj[fb].append((tb, 'line', idx))
        adj[tb].append((fb, 'line', idx))

    # Add transformer edges
    for elem_name in ['trafo', 'trafo3w', 'trafo1ph']:
        if elem_name in net and hasattr(net[elem_name], 'columns') and len(net[elem_name]) > 0:
            df = net[elem_name]
            if 'hv_bus' in df.columns and 'lv_bus' in df.columns:
                for idx, row in df.iterrows():
                    hv, lv = row['hv_bus'], row['lv_bus']
                    adj[hv].append((lv, 'trafo', idx))
                    adj[lv].append((hv, 'trafo', idx))
            if 'mv_bus' in df.columns:
                for idx, row in df.iterrows():
                    hv, mv = row['hv_bus'], row['mv_bus']
                    adj[hv].append((mv, 'trafo', idx))
                    adj[mv].append((hv, 'trafo', idx))

    # Add switch edges (bus-bus switches)
    if 'switch' in net and len(net.switch) > 0:
        for idx, row in net.switch.iterrows():
            if 'bus' in row.index and 'element' in row.index and row.get('et') == 'b':
                adj[row['bus']].append((row['element'], 'switch', idx))
                adj[row['element']].append((row['bus'], 'switch', idx))

    source_buses = set()
    if 'ext_grid_sequence' in net and len(net.ext_grid_sequence) > 0:
        source_buses = set(net.ext_grid_sequence['bus'].values)
    elif 'ext_grid' in net and len(net.ext_grid) > 0:
        source_buses = set(net.ext_grid['bus'].values)

    dist = {}
    queue = deque()
    for sb in source_buses:
        dist[sb] = 0.0
        queue.append(sb)

    while queue:
        bus = queue.popleft()
        for neighbor, edge_type, edge_idx in adj[bus]:
            if neighbor not in dist:
                if edge_type == 'line':
                    d = line_geodesic_length(net, edge_idx, bus_coords)
                elif bus in bus_coords and neighbor in bus_coords:
                    la, lo = bus_coords[bus]
                    lb, lb_lon = bus_coords[neighbor]
                    d = haversine(la, lo, lb, lb_lon)
                else:
                    d = 0.0
                dist[neighbor] = dist[bus] + d
                queue.append(neighbor)

    total_buses = len(net.bus.index.get_level_values(0).unique()) if isinstance(net.bus.index, pd.MultiIndex) else len(net.bus)
    print(f"  BFS reached {len(dist)}/{total_buses} unique buses")
    return dist

def build_bus_element_labels(net):
    """Build bus_id -> set of element letter codes for annotation."""
    labels = {}
    # Transformers
    for tname in ['trafo', 'trafo3w']:
        if tname in net and hasattr(net[tname], 'columns') and len(net[tname]) > 0:
            for col in ('hv_bus', 'lv_bus', 'mv_bus'):
                if col in net[tname].columns:
                    for b in net[tname][col].unique():
                        labels.setdefault(b, set()).add('T')
    if 'trafo1ph' in net and len(net.trafo1ph) > 0:
        tp = net.trafo1ph
        if 'bus' in tp.index.names:
            for b in tp.index.get_level_values('bus').unique():
                labels.setdefault(b, set()).add('T')
        for col in ('hv_bus', 'lv_bus'):
            if col in tp.columns:
                for b in tp[col].unique():
                    labels.setdefault(b, set()).add('T')
    # Other elements
    for tbl_name, letter in [('asymmetric_load', 'L'), ('asymmetric_sgen', 'G'),
                              ('ext_grid_sequence', 'E'), ('ext_grid', 'E'),
                              ('asymmetric_shunt', 'Sh'), ('switch', 'S')]:
        tbl = getattr(net, tbl_name, None)
        if tbl is not None and hasattr(tbl, 'columns') and len(tbl) > 0 and 'bus' in tbl.columns:
            for b in tbl['bus'].unique():
                labels.setdefault(b, set()).add(letter)
    return labels

def plot_voltage_vs_distance(net, title):
    """Plot SVVOLTAGE_V_A/B/C_PU vs distance from source for all buses with element annotations."""
    import plotly.graph_objects as go

    bus_dist = compute_distance_from_source(net)
    bus_labels = build_bus_element_labels(net)

    phase_cols = {
        'A': 'SVVOLTAGE_V_A_PU',
        'B': 'SVVOLTAGE_V_B_PU',
        'C': 'SVVOLTAGE_V_C_PU',
    }

    seen_buses = set()
    records = []
    for idx in net.res_bus.index:
        bus_num = idx[0] if isinstance(idx, tuple) else idx
        if bus_num in seen_buses:
            continue
        seen_buses.add(bus_num)

        if bus_num not in bus_dist:
            continue

        d = bus_dist[bus_num]
        row_data = net.res_bus.loc[idx]
        label = ','.join(sorted(bus_labels.get(bus_num, set())))

        for phase, col in phase_cols.items():
            if col in net.res_bus.columns:
                v = row_data[col]
                if pd.notna(v) and v != 0:
                    records.append({'bus': bus_num, 'phase': phase, 'distance_km': d, 'voltage_pu': v, 'element': label})

    df = pd.DataFrame(records)
    if df.empty:
        print(f"No data for {title}")
        return

    fig = px.scatter(
        df, x='distance_km', y='voltage_pu', color='phase',
        title=f'{title} — Phase Voltage (p.u.) vs Distance from Source',
        labels={'distance_km': 'Distance from Source (km)', 'voltage_pu': 'Voltage (p.u.)'},
        opacity=0.6,
        color_discrete_map={'A': '#e41a1c', 'B': '#377eb8', 'C': '#4daf4a'},
        hover_data=['element', 'bus'],
    )

    # Add tiny annotations for non-plain buses (only phase A to avoid clutter)
    ann_df = df[(df['element'] != '') & (df['phase'] == 'A')].drop_duplicates(subset='bus')
    for _, r in ann_df.iterrows():
        fig.add_annotation(
            x=r['distance_km'], y=r['voltage_pu'],
            text=r['element'], showarrow=False,
            font=dict(size=7, color='gray'),
            yshift=8,
        )

    fig.update_layout(height=800, template='plotly_dark')
    fig.show()

plot_voltage_vs_distance(CKT_114_16955, 'CKT_114_16955')
plot_voltage_vs_distance(CKT_4799_01085, 'CKT_4799_01085')

  BFS reached 372/454 unique buses


  BFS reached 1130/1310 unique buses


In [9]:
add_calculated_results(CKT_114_16955)
add_calculated_results(CKT_4799_01085)

This pandapower network includes the following parameter tables:
   - bus (5240 elements)
   - ext_grid_sequence (3 elements)
   - asymmetric_load (210 elements)
   - asymmetric_sgen (371 elements)
   - line (750 elements)
   - switch (1177 elements)
   - trafo1ph (420 elements)
   - line_geodata (750 elements)
   - bus_geodata (4260 elements)
   - asymmetric_shunt (9 elements)
 and the following results tables:
   - res_bus (3930 elements)
   - res_line (750 elements)
   - res_trafo (420 elements)

In [19]:
import csv
import os
import math
import numpy as np
import pandas as pd

from opendss.pf.powerflow import run_pf as dss_run_pf    

def add_dss_results(dss_file, circuit_name):
    # Run OpenDSS power flow
    pf = dss_run_pf(str(dss_file))
    d = pf['dss']

    # if not pf['converged']:
    #     print(f"WARNING: DSS power flow did not converge for {circuit_name}")

    PRECISION = 7

    # ---- Build per-bus power maps from element iteration ----
    def _collect_element_powers(element_class, element_prefix):
        """Iterate DSS elements, return {bus_name_lower: {phase: [p_kw, q_kvar]}}."""
        power_map = {}
        elem_api = getattr(d, element_class, None)
        if elem_api is None:
            return power_map
        try:
            i = elem_api.first()
        except Exception:
            return power_map
        while i != 0:
            ename = elem_api.name
            d.circuit.set_active_element(f"{element_prefix}.{ename}")
            bus_names = d.cktelement.bus_names
            powers = d.cktelement.powers          # [P1, Q1, P2, Q2, ...]
            n = d.cktelement.num_conductors
            if bus_names:
                parts = bus_names[0].split('.')
                bname = parts[0].lower()
                phases = [int(p) for p in parts[1:]] if len(parts) > 1 else list(range(1, n + 1))
                if bname not in power_map:
                    power_map[bname] = {}
                for j, ph in enumerate(phases):
                    if j * 2 + 1 < len(powers):
                        if ph not in power_map[bname]:
                            power_map[bname][ph] = [0.0, 0.0]
                        power_map[bname][ph][0] += powers[j * 2]
                        power_map[bname][ph][1] += powers[j * 2 + 1]
            i = elem_api.next()
        return power_map

    load_map = _collect_element_powers('loads', 'Load')
    gen_map  = _collect_element_powers('generators', 'Generator')
    cap_map  = _collect_element_powers('capacitors', 'Capacitor')

    # Phase number → letter mapping
    PHASE_LETTER = {1: 'A', 2: 'B', 3: 'C'}

    num_buses = d.circuit.num_buses
    rows = []

    for i in range(num_buses):
        d.circuit.set_active_bus_i(i)
        name   = d.bus.name
        kv_base = d.bus.kv_base              # L-N kV
        nodes  = list(d.bus.nodes)            # e.g. [1, 2, 3]
        va_pu  = d.bus.vmag_angle_pu          # [mag1, ang1, mag2, ang2, ...]

        # Per-phase voltage magnitudes (pu) and angles (deg)
        v_pu  = {}
        v_ang = {}
        for j, node in enumerate(nodes):
            if j * 2 + 1 < len(va_pu):
                v_pu[node]  = round(va_pu[j * 2],     PRECISION)
                v_ang[node] = round(va_pu[j * 2 + 1], PRECISION)

        # Total per-bus power (sum across phases), kW → MW
        name_lower = name.lower()
        lp = load_map.get(name_lower, {})
        gp = gen_map.get(name_lower, {})
        cp = cap_map.get(name_lower, {})

        pload = round(sum(v[0] for v in lp.values()) / 1000.0, PRECISION)
        qload = round(sum(v[1] for v in lp.values()) / 1000.0, PRECISION)
        pgen  = round(sum(v[0] for v in gp.values()) / 1000.0, PRECISION)
        qgen  = round(sum(v[1] for v in gp.values()) / 1000.0, PRECISION)
        pcap  = round(sum(v[0] for v in cp.values()) / 1000.0, PRECISION)
        qcap  = round(sum(v[1] for v in cp.values()) / 1000.0, PRECISION)

        # Per-phase load and gen power (kW → MW, kVAR → MVAR)
        per_phase_load = {}
        per_phase_gen = {}
        for ph, letter in PHASE_LETTER.items():
            lph = lp.get(ph, [0.0, 0.0])
            gph = gp.get(ph, [0.0, 0.0])
            per_phase_load[f'PLOAD_{letter}'] = round(lph[0] / 1000.0, PRECISION)
            per_phase_load[f'QLOAD_{letter}'] = round(lph[1] / 1000.0, PRECISION)
            per_phase_gen[f'PGEN_{letter}']   = round(gph[0] / 1000.0, PRECISION)
            per_phase_gen[f'QGEN_{letter}']   = round(gph[1] / 1000.0, PRECISION)

        # Per-phase injection currents: I = conj(S / V)
        i_mag = {}
        i_ang_d = {}
        for phase in [1, 2, 3]:
            if phase in v_pu and v_pu[phase] > 0 and kv_base > 0:
                v_actual_v = v_pu[phase] * kv_base * 1000.0
                v_rad = np.deg2rad(v_ang[phase])
                V_c = v_actual_v * np.exp(1j * v_rad)
                p_w  = ((gp.get(phase, [0, 0])[0] - lp.get(phase, [0, 0])[0]
                         + cp.get(phase, [0, 0])[0]) * 1000.0)
                q_va = ((gp.get(phase, [0, 0])[1] - lp.get(phase, [0, 0])[1]
                         + cp.get(phase, [0, 0])[1]) * 1000.0)
                if abs(V_c) > 0:
                    I_c = np.conj((p_w + 1j * q_va) / V_c)
                    i_mag[phase]  = round(float(abs(I_c)), PRECISION)
                    i_ang_d[phase] = round(float(np.rad2deg(np.angle(I_c))), PRECISION)

        kv_ll = round(kv_base * math.sqrt(3), PRECISION) if kv_base > 0 else 0

        row = {
            'CKT_KEY': circuit_name,
            'NODE_ID': name,
            'BASE_VOLTAGE': kv_ll,
            'PGEN': pgen, 'QGEN': qgen,
            'PGEN_A': per_phase_gen['PGEN_A'], 'PGEN_B': per_phase_gen['PGEN_B'], 'PGEN_C': per_phase_gen['PGEN_C'],
            'QGEN_A': per_phase_gen['QGEN_A'], 'QGEN_B': per_phase_gen['QGEN_B'], 'QGEN_C': per_phase_gen['QGEN_C'],
            'PCAP': pcap, 'QCAP': qcap,
            'PLOAD': pload, 'QLOAD': qload,
            'PLOAD_A': per_phase_load['PLOAD_A'], 'PLOAD_B': per_phase_load['PLOAD_B'], 'PLOAD_C': per_phase_load['PLOAD_C'],
            'QLOAD_A': per_phase_load['QLOAD_A'], 'QLOAD_B': per_phase_load['QLOAD_B'], 'QLOAD_C': per_phase_load['QLOAD_C'],
            'VA': v_pu.get(1, None),  'VA_ANGLE': v_ang.get(1, None),
            'VB': v_pu.get(2, None),  'VB_ANGLE': v_ang.get(2, None),
            'VC': v_pu.get(3, None),  'VC_ANGLE': v_ang.get(3, None),
            'IA': i_mag.get(1, None), 'IA_ANGLE': i_ang_d.get(1, None),
            'IB': i_mag.get(2, None), 'IB_ANGLE': i_ang_d.get(2, None),
            'IC': i_mag.get(3, None), 'IC_ANGLE': i_ang_d.get(3, None),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    return df

In [31]:
import opendss.tools.dss_converter as odc

dss_file = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\dss\{CKT_114_16955.CIRCUIT_KEY}.dss"
dss_net = odc.mc_net_to_opendss(CKT_114_16955, str(dss_file))
dss_df = add_dss_results(dss_file, CKT_114_16955.CIRCUIT_KEY)
dss_df

dss_file2 = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\dss\{CKT_4799_01085.CIRCUIT_KEY}.dss"
dss_net2 = odc.mc_net_to_opendss(CKT_4799_01085, str(dss_file2))
dss_df2 = add_dss_results(dss_file2, CKT_4799_01085.CIRCUIT_KEY)
dss_df2

,CKT_KEY,NODE_ID,BASE_VOLTAGE,PGEN,QGEN,PGEN_A,PGEN_B,PGEN_C,QGEN_A,QGEN_B,...,VB,VB_ANGLE,VC,VC_ANGLE,IA,IA_ANGLE,IB,IB_ANGLE,IC,IC_ANGLE
0,CKT_4799_01085,-88888814689353,12.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,1.030000,-120.000000,1.030000,120.000000,0.000000e+00,-0.000000,0.0,180.0,0.000000,0.000000
1,CKT_4799_01085,161559177,12.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,1.029998,-120.000376,1.029996,119.999568,0.000000e+00,-0.000000,0.0,180.0,0.000000,0.000000
2,CKT_4799_01085,166677255,12.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,1.029856,-120.022426,NaN,NaN,0.000000e+00,-0.000000,0.0,180.0,NaN,NaN
3,CKT_4799_01085,42808748,12.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,1.029775,-120.021611,NaN,NaN,0.000000e+00,-0.000000,0.0,180.0,NaN,NaN
4,CKT_4799_01085,58758112,12.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,NaN,NaN,1.029369,119.977089,NaN,NaN,NaN,NaN,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1281,CKT_4799_01085,-99999942734046,0.138568,-0.000868,-0.000538,-0.000868,0.0,0.0,-0.000538,0.0,...,NaN,NaN,NaN,NaN,1.371489e+03,152.112276,NaN,NaN,NaN,NaN
1282,CKT_4799_01085,-99999942807357,0.138568,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,NaN,NaN,NaN,NaN,5.593303e+01,152.181433,NaN,NaN,NaN,NaN
1283,CKT_4799_01085,-99999942807364,0.138568,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,NaN,NaN,NaN,NaN,4.298785e+01,152.224194,NaN,NaN,NaN,NaN
1284,CKT_4799_01085,-99999970903954,0.138568,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,NaN,NaN,NaN,NaN,4.000000e-07,179.974134,NaN,NaN,NaN,NaN


In [25]:
def plot_dss_voltage_vs_distance(net, dss_df, title):
    """Plot DSS VA/VB/VC (p.u.) vs distance from source using pandapower network topology."""
    bus_dist = compute_distance_from_source(net)

    # Build name -> bus_num lookup from pandapower net
    name_to_bus = {}
    for idx in net.bus.index:
        bus_num = idx[0] if isinstance(idx, tuple) else idx
        if bus_num not in name_to_bus:
            name_val = net.bus.at[idx, 'name']
            name_to_bus[str(name_val).lower()] = bus_num

    phase_cols = {'A': 'VA', 'B': 'VB', 'C': 'VC'}
    records = []
    for _, row in dss_df.iterrows():
        node_id = str(row['NODE_ID']).lower()
        bus_num = name_to_bus.get(node_id)
        if bus_num is None or bus_num not in bus_dist:
            continue
        d = bus_dist[bus_num]
        for phase, col in phase_cols.items():
            v = row.get(col)
            if pd.notna(v) and v != 0:
                records.append({'bus': bus_num, 'node_id': row['NODE_ID'], 'phase': phase,
                                'distance_km': d, 'voltage_pu': v})

    df = pd.DataFrame(records)
    if df.empty:
        print(f"No DSS data matched for {title}")
        return

    fig = px.scatter(
        df, x='distance_km', y='voltage_pu', color='phase',
        title=f'{title} — DSS Phase Voltage (p.u.) vs Distance from Source',
        labels={'distance_km': 'Distance from Source (km)', 'voltage_pu': 'Voltage (p.u.)'},
        opacity=0.6,
        color_discrete_map={'A': '#e41a1c', 'B': '#377eb8', 'C': '#4daf4a'},
        hover_data=['node_id'],
    )
    fig.update_layout(height=500, template='plotly_dark')
    fig.show()

plot_dss_voltage_vs_distance(CKT_114_16955, dss_df, 'CKT_114_16955')
plot_dss_voltage_vs_distance(CKT_4799_01085, dss_df2, 'CKT_4799_01085')

  BFS reached 372/454 unique buses


  BFS reached 1130/1310 unique buses


In [36]:
def get_dss_line_flows(dss_file):
    """Run DSS power flow and extract per-line, per-phase active and reactive power flows."""
    pf = dss_run_pf(str(dss_file))
    d = pf['dss']
    PHASE_LETTER = {1: 'A', 2: 'B', 3: 'C'}
    rows = []
    i = d.lines.first()
    while i != 0:
        lname = d.lines.name
        d.circuit.set_active_element(f"Line.{lname}")
        bus_names = d.cktelement.bus_names
        powers = d.cktelement.powers  # [P1_from, Q1_from, P2_from, Q2_from, ..., P1_to, Q1_to, ...]
        n = d.cktelement.num_conductors

        from_bus = bus_names[0].split('.')[0].lower() if bus_names else None
        to_bus = bus_names[1].split('.')[0].lower() if len(bus_names) > 1 else None

        # Parse phases from bus name
        from_parts = bus_names[0].split('.') if bus_names else []
        phases = [int(p) for p in from_parts[1:]] if len(from_parts) > 1 else list(range(1, n + 1))

        row = {'line_name': lname, 'from_bus': from_bus, 'to_bus': to_bus}
        for j, ph in enumerate(phases):
            letter = PHASE_LETTER.get(ph)
            if letter is None:
                continue
            # From-side powers
            if j * 2 + 1 < len(powers):
                row[f'P_from_{letter}_kW'] = powers[j * 2]
                row[f'Q_from_{letter}_kVAR'] = powers[j * 2 + 1]
            # To-side powers (offset by n conductors)
            to_offset = n * 2 + j * 2
            if to_offset + 1 < len(powers):
                row[f'P_to_{letter}_kW'] = powers[to_offset]
                row[f'Q_to_{letter}_kVAR'] = powers[to_offset + 1]
        rows.append(row)
        i = d.lines.next()
    return pd.DataFrame(rows)

def plot_branch_flow_vs_distance(net, dss_file, title):
    """Plot MC and DSS active/reactive branch flow on a single plot per metric, color-coded by engine."""
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go

    bus_dist = compute_distance_from_source(net)

    # Build name -> bus_num lookup
    name_to_bus = {}
    for idx in net.bus.index:
        bus_num = idx[0] if isinstance(idx, tuple) else idx
        if bus_num not in name_to_bus:
            name_to_bus[str(net.bus.at[idx, 'name']).lower()] = bus_num

    phase_map = {1: 'A', 2: 'B', 3: 'C'}

    # Engine colors
    engine_colors = {'MC': '#FF0000', 'DSS': '#FFFF00'}

    # ---- MC branch flows ----
    mc_records = []
    for idx in net.res_line.index:
        line_num = idx[0] if isinstance(idx, tuple) else idx
        phase_num = idx[1] if isinstance(idx, tuple) else None

        line_row = net.line.loc[line_num]
        fb = line_row['from_bus'] if not hasattr(line_row['from_bus'], 'iloc') else line_row['from_bus'].iloc[0]
        if fb not in bus_dist:
            continue
        d = bus_dist[fb]

        letter = phase_map.get(phase_num)
        if letter is None:
            continue

        res_row = net.res_line.loc[idx]
        p_from = res_row.get('p_from_mw', np.nan)
        q_from = res_row.get('q_from_mvar', np.nan)
        if pd.notna(p_from):
            mc_records.append({'line': line_num, 'phase': letter, 'distance_km': d,
                               'P_MW': p_from, 'Q_MVAR': q_from if pd.notna(q_from) else 0})

    mc_df = pd.DataFrame(mc_records)

    # ---- DSS branch flows ----
    dss_line_df = get_dss_line_flows(dss_file)
    dss_records = []
    for _, row in dss_line_df.iterrows():
        fb_name = row.get('from_bus')
        bus_num = name_to_bus.get(fb_name)
        if bus_num is None or bus_num not in bus_dist:
            continue
        d = bus_dist[bus_num]
        for letter in ['A', 'B', 'C']:
            p_col = f'P_from_{letter}_kW'
            q_col = f'Q_from_{letter}_kVAR'
            if p_col in row and pd.notna(row[p_col]):
                dss_records.append({'line': row['line_name'], 'phase': letter, 'distance_km': d,
                                    'P_MW': row[p_col] / 1000.0,
                                    'Q_MVAR': (row[q_col] / 1000.0) if q_col in row and pd.notna(row[q_col]) else 0})
    dss_df_lines = pd.DataFrame(dss_records)

    # ---- Single plot with 2 rows: P (top) and Q (bottom) ----
    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=[f'{title} — Active Power (P)',
                                        f'{title} — Reactive Power (Q)'],
                        vertical_spacing=0.12)

    def add_engine_scatter(df, value_col, row_num, engine_name, show_legend):
        if df.empty:
            return
        fig.add_trace(go.Scatter(
            x=df['distance_km'], y=df[value_col],
            mode='markers', name=engine_name,
            marker=dict(size=3, color=engine_colors[engine_name], opacity=0.5),
            legendgroup=engine_name, showlegend=show_legend,
        ), row=row_num, col=1)

    add_engine_scatter(mc_df, 'P_MW', 1, 'MC', True)
    add_engine_scatter(dss_df_lines, 'P_MW', 1, 'DSS', True)
    add_engine_scatter(mc_df, 'Q_MVAR', 2, 'MC', False)
    add_engine_scatter(dss_df_lines, 'Q_MVAR', 2, 'DSS', False)

    fig.update_xaxes(title_text='Distance from Source (km)')
    fig.update_yaxes(title_text='P (MW)', row=1, col=1)
    fig.update_yaxes(title_text='Q (MVAR)', row=2, col=1)
    fig.update_layout(height=700, template='plotly_dark',
                      title_text=f'{title} — Branch Flow vs Distance from Source')
    fig.show()

plot_branch_flow_vs_distance(CKT_114_16955, dss_file, 'CKT_114_16955')
plot_branch_flow_vs_distance(CKT_4799_01085, dss_file2, 'CKT_4799_01085')

  BFS reached 372/454 unique buses


  BFS reached 1130/1310 unique buses


In [41]:

with open('c:\\tmp\\st_charles_geo.pkl', 'wb') as file:
    pickle.dump(CKT_114_16955, file)


with open('c:\\tmp\\barcelona_geo.pkl', 'wb') as file:
    pickle.dump(CKT_4799_01085, file)

In [42]:
%pip install streamlit-keplergl

  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ----------------------- ---------------- 5.2/9.1 MB 45.3 MB/s eta 0:00:01
   ---------------------------------------- 9.1/9.1 MB 46.9 MB/s  0:00:00
Using cached altair-6.0.0-py3-none-any.whl (795 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
Using cached gitpython-3.1.46-py3-none-any.whl (208 kB)
Using cached gitdb-4.0.12-py3-none-any.whl (62 k

In [ ]:
import streamlit as st
from streamlit_keplergl import keplergl_static
from keplergl import KeplerGl

st.write("This is a kepler.gl map in streamlit")

map_1 = KeplerGl()
keplergl_static(map_1)